# 01 · Train X-Pool baseline (MSR-VTT 9k, official recipe)
Runs every seed in `SEEDS` **in order**: train → evaluate (best/last + frame-order probes) → collect small result files into Drive `results/<run>/`. Planned seeds: 24 (done 11/13), 0, 1. ~6.1 h per seed.

**If Colab disconnects, just re-run all cells.** Finished seeds/steps are skipped and training resumes from the last finished epoch (`resume_state.pth` on Drive). If training crashes inside a session it is resumed automatically (up to 3 attempts).

In [ ]:
# ==== 0. Settings (edit if needed) ====
DRIVE_DIR  = "/content/drive/MyDrive/컴비 프로젝트"  # Google Drive folder (persistent: data copy, checkpoints, runs)
DRIVE_ROOT = "/content/cv_drive"                   # space-free alias (symlink) of DRIVE_DIR, used in all shell commands
REPO_URL   = "https://github.com/beblack127/computervision.git"
BRANCH     = "main"
REPO       = "/content/computervision"
LOCAL_DATA = "/content/data/msrvtt"                # fast local disk (wiped when runtime ends)
CACHE_DIR  = "/content/cache/msrvtt_u12"           # frame cache (uniform 12 frames, raw uint8)
NUM_WORKERS = 8                                    # official default; keep fixed across runs

from google.colab import drive
drive.mount("/content/drive")
import os, subprocess, json, time
os.makedirs(DRIVE_DIR, exist_ok=True)
# The folder name contains a space, which would split unquoted paths in `!` shell commands -> use a symlink
if os.path.islink(DRIVE_ROOT) and os.readlink(DRIVE_ROOT) != DRIVE_DIR:
    os.remove(DRIVE_ROOT)
if not os.path.exists(DRIVE_ROOT):
    os.symlink(DRIVE_DIR, DRIVE_ROOT)
print("Drive folder:", DRIVE_DIR, "->", DRIVE_ROOT, "| contents:", sorted(os.listdir(DRIVE_ROOT)))
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!nproc && free -g | head -2 && df -h /content | tail -1

SEEDS = [0, 1]   # <- run in this order; finished seeds are skipped


In [ ]:
# ==== 1. Code: repo + pinned official X-Pool + deps ====
if not os.path.isdir(REPO):
    !git clone -b {BRANCH} {REPO_URL} {REPO}
else:
    !git -C {REPO} pull --ff-only
%cd {REPO}
!pip -q install -r requirements-colab.txt
!bash scripts/setup_xpool.sh
import sys; sys.path.insert(0, f"{REPO}/src")
!PYTHONPATH=src python -m pytest -q tests/ 2>&1 | tail -3


In [ ]:
# ==== 2. MSR-VTT videos (first time: download ~6.5GB and copy to Drive; later: copy from Drive) ====
t = time.time()
!python scripts/prepare_msrvtt.py --drive_dir {DRIVE_ROOT}/data --local_dir {LOCAL_DATA}
print(f"data ready in {time.time()-t:.0f}s")


In [ ]:
# ==== 3. Frame cache (bit-identical to official decoding) + verification on real videos ====
t = time.time()
!PYTHONPATH=src python -m tvr.frame_cache build --videos_dir {LOCAL_DATA}/videos --cache_dir {CACHE_DIR}
!PYTHONPATH=src python -m tvr.frame_cache stats --cache_dir {CACHE_DIR}
!PYTHONPATH=src python -m tvr.frame_cache verify --videos_dir {LOCAL_DATA}/videos --cache_dir {CACHE_DIR} --n 50
print(f"cache step: {time.time()-t:.0f}s")
!du -sh {CACHE_DIR}
!df -h /content | tail -1


In [ ]:
# ==== 4. For each seed: train -> evaluate -> collect (re-running skips what is already done) ====
import glob, shutil
for SEED in SEEDS:
    EXP = f'xpool_9k_s{SEED}'
    RUN_DIR = f'{DRIVE_ROOT}/runs/{EXP}'; os.makedirs(RUN_DIR, exist_ok=True)
    print(f'\n========== {EXP} ==========', flush=True)
    # -- train (official 9k hyper-parameters); model_last.pth exists only after all 5 epochs --
    for attempt in range(1, 4):
        if os.path.exists(f'{RUN_DIR}/model_last.pth'):
            break
        resume = f'--resume_state {RUN_DIR}/resume_state.pth' if os.path.exists(f'{RUN_DIR}/resume_state.pth') else ''
        print(f'[{EXP}] train attempt {attempt}, resume: {bool(resume)}', flush=True)
        !PYTHONPATH=src python -m tvr.train --exp_name {EXP} --seed {SEED} --huggingface --dataset_name MSRVTT --msrvtt_train_file 9k --cache_dir {CACHE_DIR} --num_workers {NUM_WORKERS} --batch_size 32 --noclip_lr 3e-5 --transformer_dropout 0.3 --output_dir {DRIVE_ROOT}/runs --tb_log_dir {DRIVE_ROOT}/tb {resume} 2>&1 | tee -a {RUN_DIR}/train.log | grep --line-buffered -E 'Val Epoch|R@1|Best|Saving|resume|runlog|Error|Traceback'
    if not os.path.exists(f'{RUN_DIR}/model_last.pth'):
        raise RuntimeError(f'{EXP}: training did not finish after 3 attempts, see {RUN_DIR}/train.log')
    print(f'[{EXP}] training finished', flush=True)
    # -- evaluate best (official protocol) and last, plus temporal probes on best --
    for ck in ['model_best', 'model_last']:
        if not os.path.exists(f'{RUN_DIR}/dump_{ck}/eval_original.json'):
            !PYTHONPATH=src python -m tvr.evaluate --exp_name {EXP}_eval_{ck} --huggingface --dataset_name MSRVTT --msrvtt_train_file 9k --cache_dir {CACHE_DIR} --num_workers {NUM_WORKERS} --batch_size 32 --output_dir {DRIVE_ROOT}/runs --ckpt {RUN_DIR}/{ck}.pth --dump_dir {RUN_DIR}/dump_{ck} 2>&1 | tail -20
    for order in ['reversed', 'shuffled', 'middle']:
        if not os.path.exists(f'{RUN_DIR}/dump_model_best/eval_{order}.json'):
            !PYTHONPATH=src python -m tvr.evaluate --exp_name {EXP}_eval_probe --huggingface --dataset_name MSRVTT --msrvtt_train_file 9k --cache_dir {CACHE_DIR} --num_workers {NUM_WORKERS} --batch_size 32 --output_dir {DRIVE_ROOT}/runs --ckpt {RUN_DIR}/model_best.pth --dump_dir {RUN_DIR}/dump_model_best --frame_order {order} 2>&1 | tail -20
    # -- collect small files: Drive results/<run>/ (download this folder) + repo copy --
    for dst in [f'{DRIVE_ROOT}/results/{EXP}', f'{REPO}/experiments/results/{EXP}']:
        os.makedirs(dst, exist_ok=True)
        for f in ['run_meta.json', 'eval_history.jsonl', 'train.log']:
            if os.path.exists(f'{RUN_DIR}/{f}'):
                shutil.copy(f'{RUN_DIR}/{f}', dst)
        for f in glob.glob(f'{RUN_DIR}/dump_model_best/eval_*.json'):
            shutil.copy(f, dst)
        if os.path.exists(f'{RUN_DIR}/dump_model_last/eval_original.json'):
            shutil.copy(f'{RUN_DIR}/dump_model_last/eval_original.json', f'{dst}/eval_last_original.json')
    best = json.load(open(f'{RUN_DIR}/dump_model_best/eval_original.json'))['t2v']
    last = json.load(open(f'{RUN_DIR}/dump_model_last/eval_original.json'))['t2v']
    print(f'[{EXP}] DONE  t2v R@1 best {best["R1"]} / last {last["R1"]}  ->  {DRIVE_DIR}/results/{EXP}/', flush=True)
print('\nALL SEEDS DONE:', SEEDS)
